Imports and Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.preprocessing_utils import (
    chronological_train_test_split,
    clip_outliers_iqr,
    create_sequences,
    load_feature_engineered_dataset,
    parse_datetime_index,
    scale_train_test_data,
    save_dataframe,
    split_features_and_target,
    summarize_missing_values,
)

pd.set_option("display.max_columns", None)

### Preprocessing for Modeling

This notebook prepares the feature-engineered dataset for machine learning and deep learning models.  
The preprocessing steps include missing value inspection, outlier treatment, chronological train-test splitting, feature scaling, and sequence preparation.

Load Feature Engineered Dataset

In [2]:
feature_file_path = PROJECT_ROOT / "dataset" / "feature_engineered_energy_data.csv"

energy_df = load_feature_engineered_dataset(feature_file_path)
energy_df = parse_datetime_index(energy_df, datetime_column="date")

energy_df.head()

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,RH_5,T6,RH_6,T7,RH_7,T8,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2,hour,day_of_week,month,day_of_month,is_weekend,Appliances_lag_1,Appliances_lag_3,Appliances_lag_6,Appliances_rolling_mean_6,Appliances_rolling_mean_12,T_out_RH_out_interaction,T1_RH_1_interaction,T2_RH_2_interaction
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2016-01-11 18:50:00,580,60,20.066667,46.396667,19.426667,44.400000,19.790000,44.826667,19.0,46.430000,17.10,55.000000,6.123333,87.993333,17.530000,44.263333,18.066667,48.633333,16.890000,45.290000,5.983333,734.433333,91.166667,5.833333,40.0,4.616667,8.827838,8.827838,18,0,1,11,0,230.0,60.0,50.0,176.666667,115.833333,545.480556,931.026444,862.544000
2016-01-11 19:00:00,430,50,20.133333,48.000000,19.566667,44.400000,19.890000,44.900000,19.0,46.363333,17.10,55.090000,6.123333,88.590000,17.823333,45.493333,18.066667,48.560000,16.963333,45.290000,6.000000,734.500000,91.000000,6.000000,40.0,4.600000,34.351142,34.351142,19,0,1,11,0,580.0,70.0,60.0,238.333333,146.666667,546.000000,966.400000,868.760000
2016-01-11 19:10:00,250,40,20.260000,52.726667,19.730000,45.100000,19.890000,45.493333,19.0,47.223333,17.10,55.163333,6.067500,88.215000,17.963333,46.160000,18.033333,48.666667,16.890000,45.326667,6.000000,734.616667,90.500000,6.000000,40.0,4.516667,19.205186,19.205186,19,0,1,11,0,430.0,230.0,60.0,270.000000,162.500000,543.000000,1068.242267,889.823000
2016-01-11 19:20:00,100,10,20.426667,55.893333,19.856667,45.833333,20.033333,47.526667,19.0,48.696667,17.10,55.500000,5.900000,88.156667,17.963333,45.533333,18.100000,49.193333,16.890000,45.345000,6.000000,734.733333,90.000000,6.000000,40.0,4.433333,38.492071,38.492071,19,0,1,11,0,250.0,580.0,60.0,276.666667,166.666667,540.000000,1141.714489,910.097222
2016-01-11 19:30:00,100,10,20.566667,53.893333,20.033333,46.756667,20.100000,48.466667,19.0,48.490000,17.15,56.042500,5.800000,88.366667,17.890000,44.926667,18.150000,49.200000,16.890000,45.326667,6.000000,734.850000,89.500000,6.000000,40.0,4.350000,24.884962,24.884962,19,0,1,11,0,100.0,430.0,70.0,281.666667,170.833333,537.000000,1108.406222,936.691889


In [3]:
# checking shape and info
print("Dataset shape:", energy_df.shape)
energy_df.info()

Dataset shape: (19724, 41)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 19724 entries, 2016-01-11 18:50:00 to 2016-05-27 18:00:00
Data columns (total 41 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Appliances                  19724 non-null  int64  
 1   lights                      19724 non-null  int64  
 2   T1                          19724 non-null  float64
 3   RH_1                        19724 non-null  float64
 4   T2                          19724 non-null  float64
 5   RH_2                        19724 non-null  float64
 6   T3                          19724 non-null  float64
 7   RH_3                        19724 non-null  float64
 8   T4                          19724 non-null  float64
 9   RH_4                        19724 non-null  float64
 10  T5                          19724 non-null  float64
 11  RH_5                        19724 non-null  float64
 12  T6                        

Outlier Treatment
- (missing value checking part skipped as it was previously done with the feature-engineered dataset)

In [4]:
outlier_columns = [
    "Appliances",
    "lights",
    "T_out",
    "RH_out",
    "Windspeed",
]

energy_df = clip_outliers_iqr(
    energy_df=energy_df,
    columns=outlier_columns,
    iqr_multiplier=1.5,
)

energy_df.head()

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,T5,RH_5,T6,RH_6,T7,RH_7,T8,RH_8,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2,hour,day_of_week,month,day_of_month,is_weekend,Appliances_lag_1,Appliances_lag_3,Appliances_lag_6,Appliances_rolling_mean_6,Appliances_rolling_mean_12,T_out_RH_out_interaction,T1_RH_1_interaction,T2_RH_2_interaction
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2016-01-11 18:50:00,175,0,20.066667,46.396667,19.426667,44.400000,19.790000,44.826667,19.0,46.430000,17.10,55.000000,6.123333,87.993333,17.530000,44.263333,18.066667,48.633333,16.890000,45.290000,5.983333,734.433333,91.166667,5.833333,40.0,4.616667,8.827838,8.827838,18,0,1,11,0,230.0,60.0,50.0,176.666667,115.833333,545.480556,931.026444,862.544000
2016-01-11 19:00:00,175,0,20.133333,48.000000,19.566667,44.400000,19.890000,44.900000,19.0,46.363333,17.10,55.090000,6.123333,88.590000,17.823333,45.493333,18.066667,48.560000,16.963333,45.290000,6.000000,734.500000,91.000000,6.000000,40.0,4.600000,34.351142,34.351142,19,0,1,11,0,580.0,70.0,60.0,238.333333,146.666667,546.000000,966.400000,868.760000
2016-01-11 19:10:00,175,0,20.260000,52.726667,19.730000,45.100000,19.890000,45.493333,19.0,47.223333,17.10,55.163333,6.067500,88.215000,17.963333,46.160000,18.033333,48.666667,16.890000,45.326667,6.000000,734.616667,90.500000,6.000000,40.0,4.516667,19.205186,19.205186,19,0,1,11,0,430.0,230.0,60.0,270.000000,162.500000,543.000000,1068.242267,889.823000
2016-01-11 19:20:00,100,0,20.426667,55.893333,19.856667,45.833333,20.033333,47.526667,19.0,48.696667,17.10,55.500000,5.900000,88.156667,17.963333,45.533333,18.100000,49.193333,16.890000,45.345000,6.000000,734.733333,90.000000,6.000000,40.0,4.433333,38.492071,38.492071,19,0,1,11,0,250.0,580.0,60.0,276.666667,166.666667,540.000000,1141.714489,910.097222
2016-01-11 19:30:00,100,0,20.566667,53.893333,20.033333,46.756667,20.100000,48.466667,19.0,48.490000,17.15,56.042500,5.800000,88.366667,17.890000,44.926667,18.150000,49.200000,16.890000,45.326667,6.000000,734.850000,89.500000,6.000000,40.0,4.350000,24.884962,24.884962,19,0,1,11,0,100.0,430.0,70.0,281.666667,170.833333,537.000000,1108.406222,936.691889


##### Outlier Treatment
- Outliers were treated using IQR-based clipping for selected columns rather than row deletion.  
- This approach preserves the time-series continuity while reducing the influence of extreme values.

Chronogical train-test split

In [5]:
train_df, test_df = chronological_train_test_split(
    energy_df=energy_df,
    train_ratio=0.8,
)

print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

Training set shape: (15779, 41)
Testing set shape: (3945, 41)


Split features and target

In [6]:
TARGET_COLUMN = "Appliances"

x_train_df, y_train_series = split_features_and_target(train_df, TARGET_COLUMN)
x_test_df, y_test_series = split_features_and_target(test_df, TARGET_COLUMN)

print("X_train shape:", x_train_df.shape)
print("X_test shape:", x_test_df.shape)
print("y_train shape:", y_train_series.shape)
print("y_test shape:", y_test_series.shape)

X_train shape: (15779, 40)
X_test shape: (3945, 40)
y_train shape: (15779,)
y_test shape: (3945,)


Scale Features

In [7]:
x_train_scaled, x_test_scaled, fitted_scaler = scale_train_test_data(
    x_train=x_train_df,
    x_test=x_test_df,
    scaler_type="standard",
)

print("Scaled X_train shape:", x_train_scaled.shape)
print("Scaled X_test shape:", x_test_scaled.shape)

Scaled X_train shape: (15779, 40)
Scaled X_test shape: (3945, 40)


Saving splitted datasets

In [ ]:
save_dataframe(train_df, "train_feature_engineered_data.csv")
save_dataframe(test_df, "test_feature_engineered_data.csv")